In [1]:
#import requred libraries

from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio



In [2]:
#load environment variables
load_dotenv(override=True)

True

In [ ]:
#Create a function to send an email

def send_an_email():
    sg= sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("shahid9170@gmail.com")
    to_email = To("saasmail11@gmail.com")
    content = Content("text/plain", "this is an temp importaint email")
    mail = Mail(from_email, to_email, "test mail", content).get()
    response = sg.client.mail.send.post(request_body= mail)
    print(response.status_code)

send_an_email()

202


In [4]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [6]:
#Create sales agents with different instructions 
sales_agent1= Agent(name="sales_agent1", instructions= instructions1, model="gpt-4.1-mini")

sales_agent2= Agent(name="sales_agent2", instructions= instructions2, model="gpt-5-nano-2025-08-07")

sales_agent3= Agent(name="sales_agent3", instructions=instructions3, model="gpt-5-2025-08-07")

"""
Streaming Agent Output: Why, How, and What

Overview:
Instead of waiting for the entire response from the agent, you can stream the output as it's generated—just like watching a document appear one word at a time.

Traditional Approach (No Streaming):
- You ask the agent to write an email.
- The agent works.
- When it’s done, you get the entire email at once.

Streaming Approach (This Code):
- You ask the agent to write an email.
- As soon as the agent starts generating output, the text appears on your screen word-by-word (like watching someone type in a Google Doc).
- You don’t have to wait for the full email to see progress.

How Each Part Works:
- `result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")`
  - Starts the agent and begins streaming output.
- `async for event in result.stream_events():`
  - Processes each new chunk of output as it arrives.
- `if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):`
  - Checks that this event is actual text (not a status or control message).
- `print(event.data.delta, end="", flush=True)`
  - Prints each piece of text immediately as soon as it comes in.

Analogy:
Watching the output stream is like seeing someone type live into a document, rather than receiving a completed file later.

Benefits:
- Much better user experience—users can see the answer form in real time, rather than staring at a loading icon.
"""


In [7]:
#See the text appear word-by-word as it's being written (like a typewriter) No need to wait for completion

#Start the agent and begin streaming (don't wait for the full answer)
result = Runner.run_streamed(sales_agent1, "write a cold email")
#Process each piece of text as it arrives
async for event in result.stream_events():
    #Check if this piece is actual text (not a status message)
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        #Print the text as it arrives   
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC 2 Compliance with AI-Powered Automation

Hi [First Name],

I hope this message finds you well. I’m reaching out to introduce ComplAI, a SaaS platform designed to streamline SOC 2 compliance and audit preparation through AI-powered automation.

Managing SOC 2 requirements can be time-consuming and complex. ComplAI simplifies this process by continuously monitoring your controls, generating audit-ready documentation, and providing actionable insights to keep your compliance on track—all in one centralized platform.

Would you be open to a brief call next week to discuss how ComplAI can reduce your compliance workload and help you prepare for audits with confidence?

Best regards,  
[Your Full Name]  
Sales Agent | ComplAI  
[Your Contact Information]  
[Company Website]

In [8]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Simplify Your SOC 2 Compliance with AI-Powered Automation

Hi [First Name],

Ensuring SOC 2 compliance and preparing for audits can be time-consuming and complex. At ComplAI, we’ve developed an AI-powered SaaS tool that streamlines the entire process—helping your team maintain compliance effortlessly and confidently.

Our platform automates evidence collection, monitors controls continuously, and provides actionable insights to keep you audit-ready at all times.

Could we schedule a brief call next week to discuss how ComplAI can reduce your compliance workload and mitigate audit risks?

Best regards,  
[Your Full Name]  
Sales Executive | ComplAI  
[Your Email] | [Your Phone Number]  
[Company Website]


Subject: SOC 2 on autopilot—meet ComplAI

Hi [FirstName],

Quick question: how many hours did your last SOC 2 prep take? If you’re like most security teams, it felt a bit like chasing unicorns in a spreadsheet.

That’s where ComplAI comes in. It’s an AI-powered platform that 

In [9]:
sales_picker = Agent(name="sales_picker", 
instructions="You are a sales agent that picks the best cold email from a list of options.\
    imagine you are a human and you are going to pick one of the emails to a potential customer.\
    do not give any explanation, just pick the best email from the list.", 
model="gpt-5.1-2025-11-13")

In [11]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
outputs = [result.final_output for result in results]

email_list = "cold slaes email:\n\n" + "\n\n".join(outputs)

best_email = await Runner.run(sales_picker, email_list)

print(f"best email: {best_email.final_output}")

best email: Subject: Fast-track SOC 2 with AI (without the spreadsheet grind)

Hi [First Name],

SOC 2 shouldn’t slow down your team. ComplAI automates evidence collection, maps controls to your stack (AWS, GCP, Azure, Okta, Google Workspace, GitHub, Jira), and keeps you audit-ready—no manual wrangling.

What you get:
- Tailored control set and gap analysis in minutes
- Continuous evidence collection from your tools
- Real-time alerts for control drift and exceptions
- One workspace to collaborate with your auditor

If SOC 2 is on your roadmap, open to a 15-minute walkthrough next week? I can share a sample readiness report for [Company].

Best,
[Your Name]  
ComplAI  
[Email] | [Phone] | [Calendar Link]


## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [27]:

@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("shahid9170@gmail.com")
    to_email = To("saasmail11@gmail.com")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return  {"status": response.status_code}


In [29]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001949048FCE0>, strict_json_schema=True, is_enabled=True)

In [30]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name= "sales_agent1", tool_description= description)
tool2 = sales_agent2.as_tool(tool_name= "sales_agent2", tool_description= description)
tool3 = sales_agent3.as_tool(tool_name= "sales_agent3", tool_description= description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000194904B5300>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001949055D580>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'

In [31]:
#Now it's time for our manager agent to use the tools


instructions= """
    You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""
sales_manger = Agent(name= "sales_manager", instructions= instructions, tools= tools, model= "gpt-5.1-2025-11-13")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manger"):
    result = await Runner.run(sales_manger, message)

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [37]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

suject_writer = Agent(name="Email subject writer", instructions= subject_instructions, model="gpt-4o-mini")
subject_tool = suject_writer.as_tool(tool_name= "subject_writer", tool_description= "Write a subject for a cold sales email")

html_converter = Agent(name="HTMl email body converter", instructions= html_instructions, model= "gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name= "html_converter", tool_description= "Convert a text email body to an HTML email body")

In [75]:

@function_tool
def send_an_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send an email with the given subject and HTML body"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("shahid9170@gmail.com")
    to_email = To("saasmail11@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [80]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send an email with the given subject and HTML body"""
    print(f"\n{'='*50}")
    print(f"📧 SENDING EMAIL")
    print(f"Subject: {subject}")
    print(f"Body length: {len(html_body)} chars")
    print(f"Body preview: {html_body[:100]}...")
    
    try:
        # Check API key
        api_key = os.environ.get('SENDGRID_API_KEY')
        if not api_key:
            print("❌ ERROR: SENDGRID_API_KEY not found in environment variables!")
            return {"status": "error", "message": "API key not found"}
        else:
            print(f"✅ API Key found: {api_key[:10]}...{api_key[-4:]}")
        
        # Initialize SendGrid
        print("Initializing SendGrid client...")
        sg = sendgrid.SendGridAPIClient(api_key=api_key)
        print("✅ SendGrid client initialized")
        
        # Create email components
        from_email = Email("shahid9170@gmail.com")
        to_email = To("saasmail11@gmail.com")
        print(f"From: {from_email.email}")
        print(f"To: {to_email.email}")
        
        # Create content
        content = Content("text/html", html_body)
        print(f"Content type: {content.content_type}")
        
        # Create and send mail
        print("Creating Mail object...")
        mail = Mail(from_email, to_email, subject, content).get()
        print("✅ Mail object created")
        
        # Send email
        print("Sending email via SendGrid API...")
        response = sg.client.mail.send.post(request_body=mail)
        
        # Log response
        print(f"✅ Status Code: {response.status_code}")
        print(f"Response Headers: {dict(response.headers)}")
        
        if hasattr(response, 'body') and response.body:
            print(f"Response Body: {response.body}")
        
        if response.status_code in [200, 202]:
            print("✅ EMAIL SENT SUCCESSFULLY!")
            return {"status": "success", "status_code": response.status_code}
        else:
            print(f"❌ EMAIL SEND FAILED with status: {response.status_code}")
            error_msg = getattr(response, 'body', 'No error message')
            return {"status": "error", "status_code": response.status_code, "message": str(error_msg)}
            
    except Exception as e:
        print(f"❌ EXCEPTION OCCURRED!")
        print(f"Exception Type: {type(e).__name__}")
        print(f"Exception Message: {str(e)}")
        import traceback
        print(f"Traceback:\n{traceback.format_exc()}")
        return {"status": "error", "message": str(e)}
    finally:
        print(f"{'='*50}\n")

In [81]:
tools = [subject_tool, html_tool, send_an_email]
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000194906DB600>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000194906DA340>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='send_an_email', description='Send an email with the given subject and HTML body', params_

In [82]:

instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

emailed_agent= Agent(
    name="Emailer Agent",
    instructions= instructions,
    model= "gpt-4o-mini",
    handoff_description= "convert an email to HTML and send it")

In [83]:
tools = [tool1, tool2, tool3]
handoff = [emailed_agent]
print(tools)
print(handoff)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000194904B5300>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001949055D580>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}

## And now it's time for our Sales Manager - our planning agent

In [84]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

sales_manger = Agent(
    name="sales Manager",
    instructions= instructions,
    tools= tools,
    handoffs= handoff,
    model= "gpt-5.1-2025-11-13"

)
message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automate SDR"):
    result = await Runner.run(sales_manger, message)